# 07 · Built-in Tools: Web Search, Python & Files 🔍🐍📄

Some tools are so useful that **OpenAI runs them for you** — no Python function needed. Just add them to `tools`:

| Tool | What Uncle can do |
|---|---|
| `web_search` | look things up on the internet |
| `code_interpreter` | write and run Python (great for maths!) |
| **file inputs** | read a PDF or photo you send him |

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
uncle = "You are a friendly kopitiam uncle in Singapore. Speak light Singlish. Keep replies short."

## 1. Web search 🔍

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=uncle,
    tools=[{"type": "web_search"}],
    input="Any new hawker centres opening in Singapore recently?",
)

print(response.output_text)

In [ ]:
# what happened behind the scenes?
for item in response.output:
    print(item.type)

You'll see a `web_search_call` — OpenAI did the searching, then wrote the answer.

## 2. Code interpreter (the "python tool") 🐍
AI models can be bad at maths. With `code_interpreter`, Uncle **writes and runs Python** to get it right.

In [ ]:
python_tool = {"type": "code_interpreter", "container": {"type": "auto"}}

response = client.responses.create(
    model=MODEL,
    instructions=uncle + " Use the python tool for any maths.",
    tools=[python_tool],
    input="3 kopi-o peng at $1.50, 2 kaya toast sets at $4.80, 1 milo dinosaur at $3. Split equally between 3 friends.",
)

print(response.output_text)

In [ ]:
# peek at the Python code Uncle wrote
for item in response.output:
    if item.type == "code_interpreter_call":
        print(item.code)

> 💰 Each code interpreter session costs a few cents, so we only add it when we need it.

## 3. Files: send Uncle a receipt 📄
`receipt.pdf` is in your project folder — open it to have a look.

To send a file, we turn it into **base64** (a way to write any file as text).

In [ ]:
import base64

with open("receipt.pdf", "rb") as f:            # "rb" = read the raw bytes
    pdf = base64.b64encode(f.read()).decode()    # bytes -> base64 text

pdf[:60]

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=uncle,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "What did we order? What's the total?"},
            {"type": "input_file", "filename": "receipt.pdf", "file_data": "data:application/pdf;base64," + pdf},
        ],
    }],
)

print(response.output_text)

## 4. Files + Python together 🤝

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=uncle + " Use the python tool for any maths.",
    tools=[python_tool],
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Split this bill: Mei Ling had the milo dinosaur and eggs, the rest we share equally between 3."},
            {"type": "input_file", "filename": "receipt.pdf", "file_data": "data:application/pdf;base64," + pdf},
        ],
    }],
)

print(response.output_text)

> 📱 On Telegram this works with **photos** too: part 5 of `main.py` turns a photo into an `input_image` the same way.

## ✏️ Exercises
1. Ask Uncle (with `web_search`) for the opening hours of your favourite hawker centre.
2. Upload a photo of a real receipt or menu to Codespaces (drag it into the file list) and send it as:
   `{"type": "input_image", "image_url": "data:image/jpeg;base64," + photo}`
3. 🌶️ Challenge: give Uncle **all** the tools at once — `web_search`, `python_tool` and your function tools from notebook 06.

In [ ]:
# your code here